In [ ]:
Names = "Diane Umuhoza"
ID = 101286

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ISLP import load_data
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error

In [ ]:
Hitters = load_data("Hitters").dropna()
y = Hitters["Salary"]
X = pd.get_dummies(Hitters.drop(columns=["Salary"]), drop_first=True, dtype=float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)
alphas = np.logspace(-2, 5, 100)
ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5)).fit(X_train, y_train)
lasso = make_pipeline(StandardScaler(), LassoCV(cv=5, max_iter=20000, random_state=1)).fit(X_train, y_train)
linear = LinearRegression().fit(X_train, y_train)
print({"Linear_Test_MSE": mean_squared_error(y_test, linear.predict(X_test))})
print({"Ridge_Test_MSE": mean_squared_error(y_test, ridge.predict(X_test)), "Ridge_Alpha": ridge.named_steps["ridgecv"].alpha_})
print({"Lasso_Test_MSE": mean_squared_error(y_test, lasso.predict(X_test)), "Lasso_Alpha": lasso.named_steps["lassocv"].alpha_})
print(pd.Series(lasso.named_steps["lassocv"].coef_, index=X.columns).sort_values(key=np.abs, ascending=False).head(15))

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA().fit(X_scaled)
print({"Explained_Variance_Ratio": pca.explained_variance_ratio_, "Cumulative": np.cumsum(pca.explained_variance_ratio_)})
plt.figure(figsize=(8, 5))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.tight_layout()
plt.show()

In [ ]:
pls = PLSRegression(n_components=2).fit(X_train, y_train)
print({"PLS_Test_MSE": mean_squared_error(y_test, pls.predict(X_test).ravel())})